# 5 · QProp Performance Sweep

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab · ETH Zürich

Runs QProp (blade-element / vortex performance solver) for every valid configuration across the full (velocity, RPM) operating grid and extracts per-config optima. Each (V, RPM) point is a single-point subprocess call with the working directory set to the prop-file folder — no runfile, no temp directory, no path ambiguity — and hover (V = 0) automatically uses QProp's dedicated hover solver. The resulting thrust and torque surfaces T(V, ω) and Q(V, ω) are the lookup tables the free-flight integrator (NB6/NB6b) queries at every timestep. The identical flow then runs for the 10 recovered validation propellers.

**Physics inputs:** `csv/01_geometry.csv`, `csv/04_xfoil_polars.csv`, `utils/qprop.exe`, `utils/motor.mas` (and `utils/00_validation_geometry.csv`, `csv/val_04_xfoil_polars.csv` for the validation subset)

**Physics outputs:** `csv/05_qprop_results.csv` (per-config optima: hover point near launch RPM, best figure of merit, best propulsive efficiency), `csv/05_qprop_sweep.csv.gz` (the full 2-D T(V, ω), Q(V, ω) surfaces with plausibility flags), and the validation-subset equivalents `csv/val_05_qprop_results.csv`, `csv/val_05_qprop_sweep.csv.gz`

**Structure:**

1. Imports
2. Configuration — all tunable constants, paths and settings
3. Function definitions — gating, prop-file writing, QProp execution, output parsing, plausibility gates, optima extraction
4. Main code — load and gate, prop files, QProp batch, parsing, gates, optima, save, validation subset, validation report

> **Design note — no runfile:** QProp on Windows resolves runfile paths relative to its working directory. Passing command-line arguments with `cwd` set to the prop folder removes all file-path ambiguity and lets workers run in parallel safely.
>
> **Coverage note:** the RPM grid is read from `pipeline_config.QPROP_GRID` (currently 1500–6500 rpm). If you widen the launch-RPM range in `pipeline_config.py`, re-run this notebook with `QPROP_OVERWRITE_RUNS = True` so the sweep file covers the new range — NB6 refuses to extrapolate beyond it.



## 1. Imports

In [ ]:
import os
import sys

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True

import math
import re
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

BASE = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
import pipeline_config as cfg
from utils import measurements

## 2. Configuration

In [ ]:
CSV_DIR = BASE / 'csv'
PROP_DIR = BASE / 'qprop_input' / 'props'
OUTPUT_DIR = BASE / 'qprop_output'
VAL_PROP_DIR = BASE / 'qprop_input' / 'validation'
VAL_OUTPUT_DIR = BASE / 'qprop_output' / 'validation'
RESULTS_CSV = CSV_DIR / cfg.CSV_NAMES['qprop_results']
SWEEP_CSV_GZ = CSV_DIR / cfg.CSV_NAMES['qprop_sweep']
QPROP_EXE = BASE / 'utils' / 'qprop.exe'
MOTOR_FILE = BASE / 'utils' / 'motor.mas'
POLAR_CSV = CSV_DIR / cfg.CSV_NAMES['xfoil_polars']
GEOMETRY_CSV = CSV_DIR / cfg.CSV_NAMES['geometry']

STATIONS = ['hub'] + cfg.QPROP_STATION_NAMES
AERO_KEYS = ['CL0', 'CL_a', 'CLmin', 'CLmax', 'CD0', 'CD2u', 'CD2l', 'CLCD0', 'REref', 'REexp']

MIN_CONFIDENCE = cfg.QPROP_MIN_CONFIDENCE
MIN_STATIONS = cfg.QPROP_MIN_STATIONS

HOVER_RPMS = cfg.HOVER_RPMS
FLIGHT_VS = cfg.FLIGHT_VS
FLIGHT_RPMS = cfg.FLIGHT_RPMS
LAUNCH_RPM = cfg.LAUNCH_RPM
RPM_TOL = cfg.RPM_TOL
GRID = cfg.QPROP_GRID

RHO = cfg.AIR_DENSITY_KG_M3
T_MAX_N = cfg.QPROP_T_MAX_N
P_MAX_W = cfg.QPROP_P_MAX_W

MAX_WORKERS = min(os.cpu_count() or 4, cfg.QPROP_MAX_WORKERS)
QPROP_TIMEOUT = cfg.QPROP_TIMEOUT_SEC
OVERWRITE_RUNS = cfg.QPROP_OVERWRITE_RUNS

QPROP_COLS = ['V', 'rpm', 'Dbeta', 'T', 'Q', 'Pshaft', 'Volts', 'Amps', 'effmot', 'effprop', 'adv', 'CT', 'CP', 'DV', 'eff', 'Pelec', 'Pprop', 'cl_avg', 'cd_avg']
N_PERF_COLS = 19

scientific_notation_pattern = re.compile(r'[Ee][+-]?\d+')
word_pattern = re.compile(r'[a-zA-Z]{2,}')
scientific_notation_pattern_value = scientific_notation_pattern
word_pattern_value = word_pattern

print(f'Base       : {BASE}')
print(f'QProp      : {QPROP_EXE}  exists={QPROP_EXE.exists()}')
print(f'Motor      : {MOTOR_FILE}  exists={MOTOR_FILE.exists()}')
print(f'RPM range  : {cfg.RPM_MIN}–{cfg.RPM_MAX} step {cfg.RPM_STEP}  ({len(HOVER_RPMS)} points)')
print(f'Launch RPM : {LAUNCH_RPM}')
print(f'Grid pts   : {len(GRID)}  ({len(HOVER_RPMS)} hover + {len(FLIGHT_VS) * len(FLIGHT_RPMS)} flight)')
print(f'Workers    : {MAX_WORKERS}')

## 3. Function Definitions

- **count_usable(row)** — counts how many blade stations of a propeller have a usable (converged) airfoil polar; used to decide whether the prop can be simulated.
- **station_usable(row, stn)** — True when a given station has a valid polar and was not flagged as failed.
- **build_prop_text(row)** — assembles the QProp propeller input file for one configuration: the header block (name, blade count, tip and root radius, unit conversions, default fluid constants) followed by one line per usable station with radius, chord, twist and the ten aerodynamic coefficients from NB4. Returns None when fewer than the minimum number of stations are usable.
- **write_prop_files(run_df, prop_dir, label)** — writes the prop file of every runnable configuration into the given folder (skipping files that already exist) and returns the config-to-path dictionary.
- **output_is_valid(path)** — checks that a QProp output file contains at least one complete 19-column performance line with plausible velocity and RPM, i.e. that the run produced data rather than an error.
- **run_config(cid, prop_file, prop_dir, output_dir)** — runs QProp for a single propeller across the whole operating grid (one subprocess call per grid point), concatenates the raw stdout, normalises the line endings, and writes `prop_<id>_out.txt`. Skips configs whose output is already valid unless overwriting is enabled.
- **run_qprop_batch(prop_files, prop_dir, output_dir, label)** — copies the motor file next to the prop files (QProp resolves file arguments relative to its working directory), determines which configs still need running, and executes them in a parallel batch.
- **is_performance_line(line)** — distinguishes QProp's 19-column numeric performance lines from headers, labels and blade-element lines.
- **parse_file(out_file, cid, r_tip_m)** — reads one QProp output file, extracts the deduplicated (V, RPM) performance rows into a DataFrame, and adds the derived figure of merit (hover rows) and propulsive efficiency (forward-flight rows).
- **parse_all_outputs(prop_files, output_dir, radius_map, label)** — parses every output file and concatenates the full sweep table; returns the sweep, the per-config frames and the list of parse failures.
- **apply_plausibility_gates(sweep_df)** — flags each sweep row `qprop_ok` when thrust, shaft power, figure of merit and efficiency are all inside their physical ranges.
- **extract_optima(sweep_df, label)** — per configuration, extracts the hover point nearest the launch RPM (thrust, power, CT, CP, FOM), the maximum-FOM hover point, and the maximum-efficiency forward-flight point, using only plausible rows.
- **assemble_results(records, merged_table)** — builds the results table and fills configs without any plausible row with `qprop_ok = False`, so the output always has one row per input configuration.
- **chk(cond, msg)** — pass/fail assertion helper used by the validation report.


In [ ]:
def count_usable(row):

    usable_count = 0
    for station in STATIONS:
        if row[f'tier_{station}'] != 'failed' and pd.notna(row[f'{station}_CL0']):
            usable_count += 1

    return usable_count


def station_usable(row, stn):

    return row[f'tier_{stn}'] != 'failed' and pd.notna(row[f'{stn}_CL0'])


def build_prop_text(row):

    cid = int(row['config_id'])
    blades = int(row['blade_count'])
    r_tip = float(row['radius_mm']) / 1000.0
    usable = []
    for station in STATIONS:
        if station_usable(row, station):
            usable.append(station)
    if len(usable) < MIN_STATIONS:

        return None

    r_root = float(row[f'r_{usable[0]}_mm']) / 1000.0
    lines = [
        f'Prop_{cid:05d}',
        f'{blades}  {r_tip:.6f}  {r_root:.6f}',
        '',
        '0.0  6.28',
        '-0.5  1.5',
        '0.020  0.010  0.010  0.30',
        '25000  -0.50',
        '',
        '0.001  0.001  1.0',
        '0.0    0.0    0.0',
        '',
    ]
    for station in usable:
        r = float(row[f'r_{station}_mm'])
        c = float(row[f'chord_{station}_mm'])
        b = float(row[f'twist_{station}_deg'])
        a = {}
        for key in AERO_KEYS:
            a[key] = float(row[f'{station}_{key}'])
        lines.append(
            f"{r:.3f}  {c:.3f}  {b:.3f}  "
            f"{a['CL0']:.4f}  {a['CL_a']:.4f}  {a['CLmin']:.4f}  {a['CLmax']:.4f}  "
            f"{a['CD0']:.6f}  {a['CD2u']:.6f}  {a['CD2l']:.6f}  {a['CLCD0']:.4f}  "
            f"{int(a['REref'])}  {a['REexp']:.4f}"
        )

    return '\r\n'.join(lines) + '\r\n'


def write_prop_files(run_df, prop_dir, label='Prop files'):

    prop_dir.mkdir(parents=True, exist_ok=True)
    prop_files = {}
    for row_index, row in tqdm(run_df.iterrows(), total=len(run_df), desc=label):
        cid = int(row['config_id'])
        path = prop_dir / f'prop_{cid:05d}.txt'
        if not path.exists():
            text = build_prop_text(row)
            if text is None:
                continue
            path.write_bytes(text.encode('utf-8'))
        prop_files[cid] = path

    return prop_files


def output_is_valid(path):

    if not path.exists() or path.stat().st_size < 100:

        return False

    try:
        raw = path.read_bytes()
        txt = raw.replace(b'\r\n', b'\n').replace(b'\r', b'\n').decode('utf-8', errors='ignore')
    except OSError:

        return False

    for ln in txt.splitlines():
        s = ln.strip()
        if not s:
            continue
        if s.startswith('#'):
            s = s[1:].strip()
        if not s or len(s.split()) != 19:
            continue
        if word_pattern_value.search(scientific_notation_pattern_value.sub('', s)):
            continue
        try:
            parts = s.split()
            v_val = float(parts[0])
            rpm_val = float(parts[1])
            if 0.0 <= v_val <= 15.0 and rpm_val >= 500.0:

                return True
        except (ValueError, IndexError):
            pass

    return False


def run_config(cid, prop_file, prop_dir, output_dir):

    out_file = output_dir / f'prop_{cid:05d}_out.txt'
    if not OVERWRITE_RUNS and output_is_valid(out_file):

        return out_file

    cwd_str = str(prop_dir.resolve())
    qprop_str = str(QPROP_EXE.resolve())
    prop_name = prop_file.name
    motor_name = MOTOR_FILE.name

    chunks = []
    for v, rpm in GRID:
        cmd = [qprop_str, prop_name, motor_name, str(v), str(int(rpm))]
        try:
            res = subprocess.run(cmd, capture_output=True, timeout=QPROP_TIMEOUT, cwd=cwd_str)
            chunks.append(res.stdout)
        except subprocess.TimeoutExpired:
            pass
        except Exception:
            pass

    raw_all = b'\n'.join(chunks)
    normalised = raw_all.replace(b'\r\n', b'\n').replace(b'\r', b'\n')
    out_file.write_bytes(normalised)

    return out_file


def run_qprop_batch(prop_files, prop_dir, output_dir, label='QProp'):

    output_dir.mkdir(parents=True, exist_ok=True)
    motor_file_in_prop_dir = prop_dir / MOTOR_FILE.name
    if not motor_file_in_prop_dir.exists():
        shutil.copy2(MOTOR_FILE, motor_file_in_prop_dir)
        print(f'Copied motor file to {motor_file_in_prop_dir}')

    to_run = {}
    for cid, prop_file in prop_files.items():
        if OVERWRITE_RUNS or not output_is_valid(output_dir / f'prop_{cid:05d}_out.txt'):
            to_run[cid] = prop_file
    cached = len(prop_files) - len(to_run)

    print(f'To run  : {len(to_run)}')
    print(f'Cached  : {cached}')
    print(f'ETA     : ~{len(to_run) * len(GRID) * 0.05 / max(MAX_WORKERS, 1) / 60:.0f} min')

    t0 = time.time()
    if to_run:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = {}
            for cid, prop_file in to_run.items():
                futs[ex.submit(run_config, cid, prop_file, prop_dir, output_dir)] = cid
            for fut in tqdm(as_completed(futs), total=len(futs), desc=label):
                cid = futs[fut]
                try:
                    fut.result()
                except Exception as exc:
                    print(f'  ERROR config {cid}: {exc}')
    elapsed = time.time() - t0
    print(f'Done in {elapsed:.1f} s  ({elapsed / max(len(to_run), 1):.2f} s/config)')


def is_performance_line(line):

    s = line.strip()
    if not s:

        return False

    if s.startswith('#'):
        s = s[1:].strip()
    if not s:

        return False

    s_clean = scientific_notation_pattern.sub('', s)
    if word_pattern.search(s_clean):

        return False

    parts = s.split()
    if len(parts) != N_PERF_COLS:

        return False

    try:
        v_val = float(parts[0])
        rpm_val = float(parts[1])
    except ValueError:

        return False

    if not (0.0 <= v_val <= 15.0 and rpm_val >= 500.0):

        return False

    return True


def parse_file(out_file, cid, r_tip_m):

    try:
        raw = out_file.read_bytes()
    except OSError:

        return pd.DataFrame()

    text = raw.replace(b'\r\n', b'\n').replace(b'\r', b'\n').decode('utf-8', errors='ignore')

    rows = []
    seen = set()
    for ln in text.splitlines():
        if not is_performance_line(ln):
            continue
        s = ln.strip()
        if s.startswith('#'):
            s = s[1:].strip()
        parts = s.split()
        try:
            vals = []
            for part in parts:
                vals.append(float(part))
        except ValueError:
            continue
        key = (round(vals[0], 4), round(vals[1], 1))
        if key in seen:
            continue
        seen.add(key)
        rows.append(dict(zip(QPROP_COLS, vals)))

    if not rows:

        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df.insert(0, 'config_id', cid)

    disc = math.pi * r_tip_m ** 2
    T, P, V = df['T'].values, df['Pshaft'].values, df['V'].values
    valid = (T > 0) & (P > 0)
    fom = np.full(len(df), np.nan)
    eta = np.full(len(df), np.nan)
    h = valid & (V == 0.0)
    f = valid & (V > 0.0)
    if h.any():
        fom[h] = T[h] ** 1.5 / (math.sqrt(2 * RHO * disc) * P[h])
    if f.any():
        eta[f] = T[f] * V[f] / P[f]
    df['FOM'] = fom
    df['eta'] = eta

    return df


def parse_all_outputs(prop_files, output_dir, radius_map, label='Parsing'):

    sweep_frames = []
    parse_fail = []
    for cid in tqdm(sorted(prop_files), desc=label):
        out = output_dir / f'prop_{cid:05d}_out.txt'
        if not out.exists():
            parse_fail.append(cid)
            continue
        frame = parse_file(out, cid, radius_map[cid] / 1000.0)
        if frame.empty:
            parse_fail.append(cid)
        else:
            sweep_frames.append(frame)

    if sweep_frames:
        sweep_df = pd.concat(sweep_frames, ignore_index=True)
    else:
        sweep_df = pd.DataFrame()

    return sweep_df, sweep_frames, parse_fail


def apply_plausibility_gates(sweep_df):

    sweep_df['qprop_ok'] = (
        (sweep_df['T'] >= -T_MAX_N) & (sweep_df['T'] <= T_MAX_N) &
        sweep_df['Pshaft'].between(0, P_MAX_W) &
        (sweep_df['FOM'].isna() | sweep_df['FOM'].between(0.0, 1.0)) &
        (sweep_df['eta'].isna() | sweep_df['eta'].between(0.0, 1.0))
    )

    return sweep_df


def extract_optima(sweep_df, label='Optima'):

    records = []
    for cid, grp in tqdm(sweep_df.groupby('config_id'), desc=label):
        ok = grp[grp['qprop_ok']].copy()
        rec = {'config_id': int(cid), 'qprop_ok': not ok.empty}
        if ok.empty:
            records.append(rec)
            continue

        hov = ok[ok['V'] == 0.0]
        if not hov.empty:
            band = hov[hov['rpm'].between(LAUNCH_RPM - RPM_TOL, LAUNCH_RPM + RPM_TOL)]
            if not band.empty:
                ref = band.loc[(band['rpm'] - LAUNCH_RPM).abs().idxmin()]
                rec.update({
                    'T_hover': round(float(ref['T']), 5),
                    'P_hover': round(float(ref['Pshaft']), 4),
                    'RPM_hover': round(float(ref['rpm']), 1),
                    'CT_hover': round(float(ref['CT']), 6),
                    'CP_hover': round(float(ref['CP']), 6),
                })
                if pd.notna(ref['FOM']):
                    rec['FOM_hover'] = round(float(ref['FOM']), 5)
            fh = hov.dropna(subset=['FOM'])
            if not fh.empty:
                best = fh.loc[fh['FOM'].idxmax()]
                rec.update({
                    'FOM_max': round(float(best['FOM']), 5),
                    'RPM_FOM_max': round(float(best['rpm']), 1),
                    'T_FOM_max': round(float(best['T']), 5),
                    'P_FOM_max': round(float(best['Pshaft']), 4),
                })

        flt = ok[ok['V'] > 0].dropna(subset=['eta'])
        if not flt.empty:
            best = flt.loc[flt['eta'].idxmax()]
            rec.update({
                'eta_max': round(float(best['eta']), 5),
                'V_eta_max': round(float(best['V']), 2),
                'RPM_eta_max': round(float(best['rpm']), 1),
                'T_eta_max': round(float(best['T']), 5),
                'P_eta_max': round(float(best['Pshaft']), 4),
            })

        records.append(rec)

    return records


def assemble_results(records, merged_table):

    if records:
        results_df = pd.DataFrame(records).sort_values('config_id').reset_index(drop=True)
    else:
        results_df = pd.DataFrame()

    if not results_df.empty:
        present = set(results_df['config_id'].astype(int))
    else:
        present = set()
    missing = set(merged_table['config_id'].astype(int)) - present
    if missing:
        filler = pd.DataFrame({'config_id': sorted(missing), 'qprop_ok': False})
        results_df = pd.concat([results_df, filler], ignore_index=True)

    return results_df.sort_values('config_id').reset_index(drop=True)


def chk(cond, msg):

    global ok_all
    tag = 'OK  ' if cond else 'FAIL'
    print(f'  [{tag}]  {msg}')
    if not cond:
        ok_all = False

## 4. Main Code

The main code loads the NB4 polar table, gates it to the runnable configurations (confidence and usable-station thresholds), writes the QProp prop files, runs the batch across the (V, RPM) grid, parses and gates the outputs, extracts the per-config optima, and saves the results and the sweep surface. The identical flow then runs for the validation subset, using its own prop and output folders (`qprop_input/validation/`, `qprop_output/validation/`).


### 4.1 Load NB4 Data and Gate the Runnable Set

A configuration is simulated only when its polar confidence score reaches the threshold and it has at least the minimum number of usable stations.


In [ ]:
for required_path in [POLAR_CSV, GEOMETRY_CSV, QPROP_EXE, MOTOR_FILE]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

polar_df = pd.read_csv(POLAR_CSV)
geo_df = pd.read_csv(GEOMETRY_CSV)

merged = polar_df.merge(geo_df[['config_id', 'blade_count', 'radius_mm']], on='config_id', how='inner', validate='one_to_one', suffixes=('', '_geo')).sort_values('config_id').reset_index(drop=True)
if 'radius_mm_geo' in merged.columns:
    merged['radius_mm'] = merged['radius_mm_geo']
    merged.drop(columns=['radius_mm_geo'], inplace=True)

merged['n_usable'] = merged.apply(count_usable, axis=1)

run_mask = (merged['confidence_score'] >= MIN_CONFIDENCE) & (merged['n_usable'] >= MIN_STATIONS)
run_df = merged[run_mask].copy().reset_index(drop=True)
skip_df = merged[~run_mask].copy().reset_index(drop=True)

print(f'Total   : {len(merged)}')
print(f'Run     : {len(run_df)}')
print(f'Skipped : {len(skip_df)}')
print()
print(run_df['n_usable'].value_counts().sort_index().to_string())

### 4.2 Write QProp Prop Files

In [ ]:
prop_files = write_prop_files(run_df, PROP_DIR)

print(f'Prop files: {len(prop_files)}  in {PROP_DIR}')
sample_cid = next(iter(prop_files))
print(f'\nSample prop_{sample_cid:05d}.txt:')
print(prop_files[sample_cid].read_text(encoding='utf-8'))

### 4.3 Run the QProp Batch

In [ ]:
run_qprop_batch(prop_files, PROP_DIR, OUTPUT_DIR, label='QProp')

sample_out = OUTPUT_DIR / f'prop_{next(iter(prop_files)):05d}_out.txt'
if sample_out.exists():
    txt = sample_out.read_bytes().replace(b'\r\n', b'\n').replace(b'\r', b'\n').decode('utf-8', 'ignore')
    non_empty_lines = []
    for line in txt.splitlines():
        if line.strip():
            non_empty_lines.append(line)
    print(f'\nSample {sample_out.name}: {len(non_empty_lines)} non-empty lines, {sample_out.stat().st_size / 1024:.0f} KB')
    if non_empty_lines:
        print(f'  First non-empty: {non_empty_lines[0][:80]}')

### 4.4 Parse QProp Outputs

In [ ]:
radius_map = merged.set_index('config_id')['radius_mm'].to_dict()

sweep_df, sweep_frames, parse_fail = parse_all_outputs(prop_files, OUTPUT_DIR, radius_map)

print(f'Parsed OK      : {len(sweep_frames)}')
print(f'Parse failures : {len(parse_fail)}')
if not sweep_df.empty:
    print(f'Total rows     : {len(sweep_df):,}')
    print(f'Hover rows     : {(sweep_df["V"] == 0.0).sum():,}')
    print(f'Flight rows    : {(sweep_df["V"] > 0.0).sum():,}')
    print(f'Unique V vals  : {sorted(sweep_df["V"].unique())}')

### 4.5 Physical Plausibility Gates

A sweep row passes (`qprop_ok`) when the thrust magnitude, the shaft power, the figure of merit and the propulsive efficiency are all inside their physical ranges.


In [ ]:
if not sweep_df.empty:
    sweep_df = apply_plausibility_gates(sweep_df)
    n = len(sweep_df)
    nok = sweep_df['qprop_ok'].sum()
    print(f'Total rows : {n:,}')
    print(f'Plausible  : {nok:,}  ({100 * nok / n:.1f}%)')
    print(f'Flagged    : {n - nok:,}  ({100 * (n - nok) / n:.2f}%)')
else:
    print('sweep_df empty.')

### 4.6 Extract Per-Config Optima

In [ ]:
if not sweep_df.empty:
    records = extract_optima(sweep_df)
else:
    records = []

results_df = assemble_results(records, merged)

print(f'Summary rows : {len(results_df)}')
for column in ['FOM_hover', 'FOM_max', 'eta_max']:
    if column in results_df.columns:
        print(f'  {column}: {results_df[column].notna().sum()} valid configs')

n_ok = int(results_df['qprop_ok'].sum())
n_tot = len(results_df)
print()
print(f'Final rows     : {n_tot}  (expected {len(merged)})')
print(f'qprop_ok=True  : {n_ok}  ({100 * n_ok / n_tot:.1f}%)')
print(f'qprop_ok=False : {n_tot - n_ok}')

### 4.7 Save Outputs

In [ ]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(RESULTS_CSV, index=False)
print(f'Summary CSV  : {RESULTS_CSV}  ({len(results_df)} rows)')
if not sweep_df.empty:
    sweep_df.to_csv(SWEEP_CSV_GZ, index=False, compression='gzip')
    mb = SWEEP_CSV_GZ.stat().st_size / 1e6
    print(f'Sweep CSV.gz : {SWEEP_CSV_GZ}  ({mb:.1f} MB, {len(sweep_df):,} rows)')

### 4.8 Validation Subset — Recovered Geometry

Runs the identical flow for the 10 recovered validation propellers (see NB3/NB4), using their own prop-file and output folders so the main sweep data is never touched. Writes `csv/val_05_qprop_results.csv` and `csv/val_05_qprop_sweep.csv.gz`.


In [ ]:
val_polar_csv = CSV_DIR / 'val_04_xfoil_polars.csv'
val_geometry_path = measurements.validation_geometry_path(BASE)

if val_polar_csv.exists() and val_geometry_path.exists():
    val_polar_df = pd.read_csv(val_polar_csv)
    val_geo_df = measurements.load_recovered_validation_geometry(BASE)

    val_merged = val_polar_df.merge(val_geo_df[['config_id', 'blade_count', 'radius_mm']], on='config_id', how='inner', validate='one_to_one', suffixes=('', '_geo')).sort_values('config_id').reset_index(drop=True)
    if 'radius_mm_geo' in val_merged.columns:
        val_merged['radius_mm'] = val_merged['radius_mm_geo']
        val_merged.drop(columns=['radius_mm_geo'], inplace=True)
    val_merged['n_usable'] = val_merged.apply(count_usable, axis=1)

    val_run_mask = (val_merged['confidence_score'] >= MIN_CONFIDENCE) & (val_merged['n_usable'] >= MIN_STATIONS)
    val_run_df = val_merged[val_run_mask].copy().reset_index(drop=True)
    print(f'Validation subset: {len(val_merged)} total, {len(val_run_df)} runnable')

    val_prop_files = write_prop_files(val_run_df, VAL_PROP_DIR, label='Prop files (validation)')
    run_qprop_batch(val_prop_files, VAL_PROP_DIR, VAL_OUTPUT_DIR, label='QProp (validation)')

    val_radius_map = val_merged.set_index('config_id')['radius_mm'].to_dict()
    val_sweep_df, val_sweep_frames, val_parse_fail = parse_all_outputs(val_prop_files, VAL_OUTPUT_DIR, val_radius_map, label='Parsing (validation)')
    print(f'Validation parse OK: {len(val_sweep_frames)}  failures: {len(val_parse_fail)}')

    if not val_sweep_df.empty:
        val_sweep_df = apply_plausibility_gates(val_sweep_df)
        val_records = extract_optima(val_sweep_df, label='Optima (validation)')
    else:
        val_records = []
    val_results_df = assemble_results(val_records, val_merged)

    val_results_df.to_csv(CSV_DIR / 'val_05_qprop_results.csv', index=False)
    print(f'Saved -> val_05_qprop_results.csv  ({len(val_results_df)} rows)')
    if not val_sweep_df.empty:
        val_sweep_df.to_csv(CSV_DIR / 'val_05_qprop_sweep.csv.gz', index=False, compression='gzip')
        print(f'Saved -> val_05_qprop_sweep.csv.gz  ({len(val_sweep_df):,} rows)')
else:
    print('val_04_xfoil_polars.csv or the validation geometry not found — skipping the validation subset.')
    print('Run NB3 and NB4 first to generate the validation-subset inputs.')

### 4.9 Validation Report

In [ ]:
print('=' * 65)
print('VALIDATION REPORT — NB5 QProp')
print('=' * 65)
ok_all = True

chk(RESULTS_CSV.exists(), 'Results CSV written')
chk(len(results_df) == len(merged), f'Row count = {len(merged)}')
chk(results_df['config_id'].is_unique, 'config_id unique')
rb = pd.read_csv(RESULTS_CSV)
chk(rb.shape == results_df.shape, f'CSV round-trip {rb.shape}')

n_ok = int(results_df['qprop_ok'].sum())
n_tot = len(results_df)
rate = n_ok / n_tot
chk(n_ok > 0, 'At least one success')
chk(rate > 0.80, f'Success rate > 80%  (actual: {100 * rate:.1f}%)')

print()
if not sweep_df.empty:
    expected = len(GRID)
    rpc = sweep_df.groupby('config_id').size()
    full_pct = (rpc == expected).mean() * 100
    chk(rpc.median() >= 380, f'Median rows/config >= 380  (actual median={rpc.median():.0f}, min={rpc.min()}, full={full_pct:.1f}%)')
    chk((sweep_df['V'] == 0.0).sum() > 0, f'Hover rows present: {(sweep_df["V"] == 0.0).sum():,}')
else:
    chk(False, 'sweep_df empty')

print()
if 'qprop_ok' in results_df.columns:
    ok_rows = results_df[results_df['qprop_ok'] == True]
else:
    ok_rows = pd.DataFrame()
for column, lo, hi in [('FOM_hover', 0.0, 1.0), ('FOM_max', 0.0, 1.0), ('eta_max', 0.0, 1.0), ('T_hover', 0.0, T_MAX_N), ('P_hover', 0.0, P_MAX_W)]:
    if column not in ok_rows.columns:
        continue
    vals = ok_rows[column].dropna()
    if vals.empty:
        continue
    chk(vals.between(lo, hi).all(), f'{column:<14}  [{lo},{hi}]  min={vals.min():.4f}  med={vals.median():.4f}  max={vals.max():.4f}')

print()
print('=' * 65)
print('ALL CHECKS PASSED' if ok_all else 'SOME CHECKS FAILED')
print('=' * 65)
print(f'Success: {n_ok}/{n_tot}  ({100 * rate:.1f}%)')
if not sweep_df.empty:
    print(f'Sweep  : {len(sweep_df):,} pts across {len(sweep_frames)} configs')